In [1]:
import os
 
# Base dataset folder
base = "dataset"

utk = os.path.join(base, "UTKFace")
adience = os.path.join(base, "AdienceBenchmarkGenderAndAgeClassification")
adience_faces = os.path.join(adience, "faces")

print("UTKFace folder exists:", os.path.exists(utk))
print("Adience folder exists:", os.path.exists(adience))
print("Adience faces folder exists:", os.path.exists(adience_faces))

print("\nUTKFace image count:", len(os.listdir(utk)))
print("Adience faces folder count:", len(os.listdir(adience_faces)))

UTKFace folder exists: True
Adience folder exists: True
Adience faces folder exists: True

UTKFace image count: 23708
Adience faces folder count: 168


Numbers of images are there in UTK

In [ ]:
import os

utk_dir = "dataset/UTKFace"

utk_paths = []
utk_labels = []   # (age, gender)

for file in os.listdir(utk_dir):
    if file.endswith(".jpg"):
        try:
            age, gender, _ = file.split("_")[:3]
            age = int(age)
            gender = int(gender)

            utk_paths.append(os.path.join(utk_dir, file))
            utk_labels.append((age, gender))
        except:
            pass 

len(utk_paths), len(utk_labels)

(23708, 23708)

Numbers of images are there in adience

In [3]:
import os
import pandas as pd

adience_dir = "dataset/AdienceBenchmarkGenderAndAgeClassification"
faces_dir = os.path.join(adience_dir, "faces")

print("Faces folder exists:", os.path.exists(faces_dir))

adience_paths = []
adience_labels = []

for i in range(5):
    fold_file = os.path.join(adience_dir, f"fold_{i}_data.txt")
    print("\nReading:", fold_file, "Exists:", os.path.exists(fold_file))

    df = pd.read_csv(fold_file, sep="\t")

    for _, row in df.iterrows():
        user = str(row["user_id"])
        img = str(row["original_image"])
        face_id = str(row["face_id"])

        # Correct Adience filename format
        filename = f"coarse_tilt_aligned_face.{face_id}.{img}"

        img_path = os.path.join(faces_dir, user, filename)

        if os.path.exists(img_path):
            adience_paths.append(img_path)
            adience_labels.append((row["age"], row["gender"]))

print("\nFINAL RESULT:")
len(adience_paths), len(adience_labels)

Faces folder exists: True

Reading: dataset/AdienceBenchmarkGenderAndAgeClassification/fold_0_data.txt Exists: True

Reading: dataset/AdienceBenchmarkGenderAndAgeClassification/fold_1_data.txt Exists: True

Reading: dataset/AdienceBenchmarkGenderAndAgeClassification/fold_2_data.txt Exists: True

Reading: dataset/AdienceBenchmarkGenderAndAgeClassification/fold_3_data.txt Exists: True

Reading: dataset/AdienceBenchmarkGenderAndAgeClassification/fold_4_data.txt Exists: True

FINAL RESULT:


(19370, 19370)

<h1>Gender Model</h1>

In [4]:
import os
import numpy as np
import pandas as pd
import random
from tensorflow.keras import layers, models
import tensorflow as tf


2025-12-10 07:04:44.648436: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
utk_folder = "dataset/UTKFace" 

utk_paths = []
utk_genders = []

for file in os.listdir(utk_folder):
    if file.endswith(".jpg"):
        parts = file.split("_")
        if len(parts) >= 2:
            gender = int(parts[1])
            path = os.path.join(utk_folder, file)

            utk_paths.append(path)
            utk_genders.append(gender)

print("UTKFace Gender Distribution:", np.unique(utk_genders, return_counts=True))


UTKFace Gender Distribution: (array([0, 1]), array([12391, 11317]))


In [6]:
adience_dir = "dataset/AdienceBenchmarkGenderAndAgeClassification"
faces_dir = os.path.join(adience_dir, "faces")

adience_paths = []
adience_genders = []

for fold in range(5):
    fold_file = os.path.join(adience_dir, f"fold_{fold}_data.txt")

    df = pd.read_csv(fold_file, sep="\t")

    for _, row in df.iterrows():

        g = row["gender"]
        if g not in ["m", "f"]:
            continue  # skip unknown gender

        gender = 0 if g == "m" else 1  # convert to numbers

        user = str(row["user_id"])
        img = str(row["original_image"])
        face_id = str(row["face_id"])

        # correct adience filename
        filename = f"coarse_tilt_aligned_face.{face_id}.{img}"
        img_path = os.path.join(faces_dir, user, filename)

        if os.path.exists(img_path):
            adience_paths.append(img_path)
            adience_genders.append(gender)

print("Adience Gender Distribution:", np.unique(adience_genders, return_counts=True))
print("Adience Total Loaded:", len(adience_paths))


Adience Gender Distribution: (array([0, 1]), array([8120, 9372]))
Adience Total Loaded: 17492


In [7]:
all_paths = utk_paths + adience_paths
all_genders = utk_genders + adience_genders

print("Merged Gender Dist:", np.unique(all_genders, return_counts=True))


Merged Gender Dist: (array([0, 1]), array([20511, 20689]))


In [8]:
male_paths = [p for p, g in zip(all_paths, all_genders) if g == 0]
female_paths = [p for p, g in zip(all_paths, all_genders) if g == 1]

print("Total males:", len(male_paths))
print("Total females:", len(female_paths))

min_count = min(len(male_paths), len(female_paths))
print("Using balanced count:", min_count)

balanced_paths = male_paths[:min_count] + female_paths[:min_count]
balanced_labels = [0]*min_count + [1]*min_count


Total males: 20511
Total females: 20689
Using balanced count: 20511


In [9]:
combined = list(zip(balanced_paths, balanced_labels))
random.shuffle(combined)
final_paths, final_labels = zip(*combined)

final_paths = list(final_paths)
final_labels = list(final_labels)

print("Final Balanced Gender Dist:", np.unique(final_labels, return_counts=True))


Final Balanced Gender Dist: (array([0, 1]), array([20511, 20511]))


In [10]:
IMG_SIZE = 128
BATCH = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label

paths_tensor = tf.constant(final_paths)
labels_tensor = tf.constant(final_labels)

ds = tf.data.Dataset.from_tensor_slices((paths_tensor, labels_tensor))

ds = ds.shuffle(buffer_size=len(final_paths), seed=42)

train_size = int(0.8 * len(final_paths))
train_ds = ds.take(train_size).map(load_image).batch(BATCH).prefetch(tf.data.AUTOTUNE)
val_ds = ds.skip(train_size).map(load_image).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [11]:
gender_model = models.Sequential([
    layers.Input(shape=(128,128,3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(2, activation='softmax')  # male/female
])

gender_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gender_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 128, 128, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 64, 64, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 64, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 32, 32, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 32, 32, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 16, 16, 128)       0

In [12]:
print(len(train_ds))
print(len(val_ds))

1026
257


In [13]:
EPOCHS = 5

history = gender_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/5
1026/1026 [==============================] - 183s 178ms/step - loss: 0.4842 - accuracy: 0.7591 - val_loss: 0.3472 - val_accuracy: 0.8412
Epoch 2/5
1026/1026 [==============================] - 180s 175ms/step - loss: 0.3489 - accuracy: 0.8406 - val_loss: 0.3098 - val_accuracy: 0.8592
Epoch 3/5
1026/1026 [==============================] - 177s 172ms/step - loss: 0.3036 - accuracy: 0.8668 - val_loss: 0.2610 - val_accuracy: 0.8851
Epoch 4/5
1026/1026 [==============================] - 179s 174ms/step - loss: 0.2632 - accuracy: 0.8875 - val_loss: 0.2231 - val_accuracy: 0.9053
Epoch 5/5
1026/1026 [==============================] - 176s 171ms/step - loss: 0.2291 - accuracy: 0.9043 - val_loss: 0.1680 - val_accuracy: 0.9288


In [14]:
gender_model.save("gender_model.h5")
gender_model.save_weights("gender_weights.h5")
print("model & weights saved ")

/opt/conda/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


model & weights saved 


In [ ]:
IMG_SIZE = 128  

def predict_gender_from_image(img_path, model_path="gender_model.h5"):
    # Load the model
    model = tf.keras.models.load_model(model_path)

    # Load & preprocess image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = tf.expand_dims(img, axis=0)

    # Predict
    pred = model.predict(img)[0]

    # Convert to label
    gender = "Male" if np.argmax(pred) == 0 else "Female"

    # Output
    print("\nImage:", img_path)
    print("Prediction:", gender)
    print("Probabilities:", pred)

    return gender, pred
    
predict_gender_from_image("val_images/t1.png")


1/1 [==============================] - 0s 68ms/step

Image: val_images/t1.png
Prediction: Female
Probabilities: [0.19668442 0.8033155 ]


('Female', array([0.19668442, 0.8033155 ], dtype=float32))

<h1>Age Model</h1>

In [19]:
!pip install --upgrade \
    numpy==1.26.4 \
    pandas==2.1.4 \
    numexpr==2.8.8 \
    bottleneck==1.3.7 \
    tensorflow==2.14.0


  Using cached numexpr-2.8.8-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.9 kB)
  Using cached Bottleneck-1.3.7-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.5 kB)
  Using cached tensorflow-2.14.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached numexpr-2.8.8-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (377 kB)
Using cached Bottleneck-1.3.7-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (358 kB)
Using cached tensorflow-2.14.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (489.9 MB)
  Attempting uninstall: numexpr
    Found existing installation: numexpr 2.9.0
    Uninstalling numexpr-2.9.0:
      Successfully uninstalled numexpr-2.9.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.15.0
    Uninstalling tensorflow-2.15.0:╸━━━━━━━━━━━━━ 2/3 [tensor

In [20]:
!pip install numpy==1.26.4 tensorflow==2.16.1
!pip install "opencv-python<4.9.0" "opencv-python-headless<4.9.0" --force-reinstall

  Using cached tensorflow-2.16.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.3 kB)
  Using cached ml_dtypes-0.3.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached tensorboard-2.16.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached keras-3.12.0-py3-none-any.whl.metadata (5.9 kB)
Using cached tensorflow-2.16.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (589.8 MB)
Using cached ml_dtypes-0.3.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
Using cached tensorboard-2.16.2-py3-none-any.whl (5.5 MB)
Using cached keras-3.12.0-py3-none-any.whl (1.5 MB)
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.2.0
    Uninstalling ml-dtypes-0.2.0:
      Successfully uninstalled ml-dtypes-0.2.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.14.1
    Uninstalling tensorboard-2.14.1:
      Successfully uninstalled tensorboard-2.14.1
  At

In [ ]:
!pip uninstall -y numpy pandas tensorflow keras
!pip install numpy==1.26.4 pandas==2.2.3 tensorflow==2.15.0 keras==2.15.0

Found existing installation: numpy 2.3.5
Uninstalling numpy-2.3.5:
  Successfully uninstalled numpy-2.3.5
Found existing installation: pandas 2.1.4
Uninstalling pandas-2.1.4:
  Successfully uninstalled pandas-2.1.4
Found existing installation: tensorflow 2.16.1
Uninstalling tensorflow-2.16.1:
  Successfully uninstalled tensorflow-2.16.1
Found existing installation: keras 3.12.0
Uninstalling keras-3.12.0:
  Successfully uninstalled keras-3.12.0
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.4 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached ml_dtypes-0.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached tensorboard-2.15.2-py3-none-any.whl.metadata (1.7

In [22]:
!pip uninstall -y numpy pandas bottleneck numexpr


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: Bottleneck 1.3.7
Uninstalling Bottleneck-1.3.7:
  Successfully uninstalled Bottleneck-1.3.7
Found existing installation: numexpr 2.8.8
Uninstalling numexpr-2.8.8:
  Successfully uninstalled numexpr-2.8.8


In [23]:
!pip install \
  numpy==1.26.4 \
  pandas==2.1.4 \
  tensorflow-cpu==2.14.0 \
  opencv-python==4.8.0.76 \
  scipy==1.11.4 \
  numexpr==2.9.0 \
  tables==3.8.0

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.1.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached opencv_python-4.8.0.76-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached numexpr-2.9.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.9 kB)
  Using cached tensorboard-2.14.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached tensorflow_estimator-2.14.0-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached keras-2.14.0-py3-none-any.whl.metadata (2.4 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Using cached pandas-2.1.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.2 MB)
Using cached opencv_python-4.8.0.76-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (61.7 MB)
Using cached numexpr-2.9.0-cp311-cp311-manylinux_2_17_x86_64.manyli

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping


In [25]:
utk_folder = "dataset/UTKFace"

utk_paths = []
utk_ages  = []
utk_genders = []

for file in os.listdir(utk_folder):
    if not file.endswith(".jpg"):
        continue
    try:
        age, gender, _ = file.split("_")[:3]
        age = int(age)
        gender = int(gender)
        
        utk_paths.append(os.path.join(utk_folder, file))
        utk_ages.append(age)
        utk_genders.append(gender)
    except:
        pass

print("UTKFace loaded:", len(utk_paths))


UTKFace loaded: 23708


In [26]:
def utk_age_to_group(age):
    if 0 <= age <= 13:   return 0
    if 14 <= age <= 20:  return 1
    if 21 <= age <= 35:  return 2
    if 36 <= age <= 50:  return 3
    if 51 <= age <= 65:  return 4
    if 66 <= age <= 75:  return 5
    if age >= 76:        return 6
    return None

utk_age_groups = [utk_age_to_group(a) for a in utk_ages]


In [ ]:
adience_dir = "dataset/AdienceBenchmarkGenderAndAgeClassification"
faces_dir   = os.path.join(adience_dir, "faces")

adience_paths = []
adience_ages  = []
adience_genders = []

for fold in range(5):
    fold_file = os.path.join(adience_dir, f"fold_{fold}_data.txt")
    df = pd.read_csv(fold_file, sep="\t")

    for _, row in df.iterrows():
        user = str(row["user_id"])
        img = str(row["original_image"])
        face = str(row["face_id"])
        gender = row["gender"]
        age = row["age"]

        filename = f"coarse_tilt_aligned_face.{face}.{img}"
        full_path = os.path.join(faces_dir, user, filename)

        if not os.path.exists(full_path):
            continue

        adience_paths.append(full_path)
        adience_ages.append(age)
        adience_genders.append(gender)

print("Adience loaded:", len(adience_paths))


Adience loaded: 19370


In [28]:
def clean_age(age):
    if age is None or str(age) == "nan":
        return None

    age = str(age).strip()

    # formats: "(a, b)"
    if age.startswith("(") and age.endswith(")"):
        age = age.replace("(", "").replace(")", "")
        a, b = age.split(",")
        return (int(a) + int(b)) // 2
    
    # "60+"
    if "+" in age:
        return int(age.replace("+", ""))

    # "a-b"
    if "-" in age:
        a, b = age.split("-")
        return (int(a) + int(b)) // 2

    # numeric
    if age.isdigit():
        return int(age)

    return None

def clean_gender(g):
    if g == "m": return 0
    if g == "f": return 1
    return None


In [ ]:
adience_clean_paths = []
adience_age_groups = []
adience_gender_ids = []

for path, age, gender in zip(adience_paths, adience_ages, adience_genders):
    a = clean_age(age)
    g = clean_gender(gender)
    if a is None or g is None:
        continue

    group = utk_age_to_group(a)
    if group is None:
        continue

    adience_clean_paths.append(path)
    adience_age_groups.append(group)
    adience_gender_ids.append(g)

print("Clean Adience:", len(adience_clean_paths))


Clean Adience: 17452


In [ ]:
merged_paths = utk_paths + adience_clean_paths
merged_age_groups = utk_age_groups + adience_age_groups
merged_genders = utk_genders + adience_gender_ids

print("Merged total:", len(merged_paths))


Merged total: 41160


In [31]:
def merge_groups(g):
    if g in [0, 1]: return 0   # children/teens
    if g == 2:      return 1   # young adults
    if g == 3:      return 2   # adults
    if g == 4:      return 3   # middle-aged
    return 4                  # seniors (5 & 6)

final_age_groups = [merge_groups(g) for g in merged_age_groups]
print("Distribution after merging:", Counter(final_age_groups))


Distribution after merging: Counter({1: 15912, 0: 12360, 2: 7309, 3: 3014, 4: 2565})


In [32]:
group_dict = defaultdict(list)
for i, g in enumerate(final_age_groups):
    group_dict[g].append(i)

TARGET = min(len(v) for v in group_dict.values())
print("Balancing all groups to:", TARGET)


Balancing all groups to: 2565


In [33]:
balanced_paths = []
balanced_labels = []

for g, idx_list in sorted(group_dict.items()):
    chosen = np.random.choice(idx_list, TARGET, replace=True)
    for idx in chosen:
        balanced_paths.append(merged_paths[idx])
        balanced_labels.append(g)

print("Final balanced distribution:", Counter(balanced_labels))


Final balanced distribution: Counter({0: 2565, 1: 2565, 2: 2565, 3: 2565, 4: 2565})


In [ ]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    balanced_paths, balanced_labels,
    test_size=0.2, random_state=42, shuffle=True
)

print("Train images:", len(train_paths))
print("Val images:", len(val_paths))


Train images: 10260
Val images: 2565


In [ ]:
IMG_SIZE = 256
BATCH = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    .shuffle(5000)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset loaded")


Dataset loaded


In [ ]:
def age_model():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        layers.Conv2D(32, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(256, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D(),layers.Dropout(0.4),
        layers.Dropout(0.2),
        
        layers.Flatten(),

        layers.Dense(512, activation='relu'),
        layers.Dropout(0.4),

        layers.Dense(5, activation='softmax')
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

age_model = age_model()
age_model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 256, 256, 32)      896       
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 128, 128, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_4 (Conv2D)           (None, 128, 128, 64)      18496     
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 64, 64, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 64, 64, 128)       73856     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 32, 32, 128)      

In [ ]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True
)

In [38]:
EPOCHS = 7
history = age_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

Epoch 1/7
321/321 [==============================] - 406s 1s/step - loss: 1.4495 - accuracy: 0.3622 - val_loss: 1.2271 - val_accuracy: 0.4651
Epoch 2/7
321/321 [==============================] - 401s 1s/step - loss: 1.1673 - accuracy: 0.5036 - val_loss: 1.1138 - val_accuracy: 0.5419
Epoch 3/7
321/321 [==============================] - 392s 1s/step - loss: 1.0346 - accuracy: 0.5710 - val_loss: 0.9905 - val_accuracy: 0.5922
Epoch 4/7
321/321 [==============================] - 392s 1s/step - loss: 0.9272 - accuracy: 0.6199 - val_loss: 0.9358 - val_accuracy: 0.6230
Epoch 5/7
321/321 [==============================] - 390s 1s/step - loss: 0.7990 - accuracy: 0.6769 - val_loss: 0.8888 - val_accuracy: 0.6565
Epoch 6/7
321/321 [==============================] - 392s 1s/step - loss: 0.7037 - accuracy: 0.7134 - val_loss: 0.9013 - val_accuracy: 0.6561
Epoch 7/7
321/321 [==============================] - 393s 1s/step - loss: 0.6008 - accuracy: 0.7599 - val_loss: 0.8539 - val_accuracy: 0.6749


In [39]:
age_model.save("age_model.h5")
age_model.save_weights("age_weights.h5")
print("Saved model + weights.")

/opt/conda/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Saved model + weights.


In [ ]:
IMG_SIZE = 256  

def predict_age_from_image(img_path, model_path="age_model.h5"):
    
    # Load the trained model
    model = tf.keras.models.load_model(model_path)

    # Load & preprocess image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = tf.expand_dims(img, axis=0)

    # Model prediction
    pred = model.predict(img)[0]

    # Regression model (returns a single age)
    # pred = [63.7]
    if pred.shape == () or pred.shape == (1,):
        predicted_age = float(pred)
    
    # Classification model (returns probabilities)
    # pred = [0.1, 0.5, 0.3, 0.1]
    else:
        predicted_age = int(np.argmax(pred))

    # Output details
    print("\nImage:", img_path)
    print("Predicted Age:", predicted_age)
    print("Raw Model Output:", pred)

    return predicted_age, pred


# Example usage
predicted_age,_ = predict_age_from_image("val_images/t52.png")
age_groups = ["Child", "Teen", "Young Adult", "Adult", "Middle Age", "Senior"]
predicted_age_group = age_groups[predicted_age]
print("Age Group:", predicted_age_group)

1/1 [==============================] - 0s 86ms/step

Image: val_images/t52.png
Predicted Age: 2
Raw Model Output: [0.05246735 0.20026956 0.7377003  0.00078182 0.00878102]
Age Group: Young Adult
